# 01 — Underlying Price Model · Task 14.2

This notebook samples the four price models that drive Inflexion's quant work (`spec.md` §9).

Four models, increasing in realism:

1. **GBM** — the textbook baseline (lognormal terminal, thin tails).
2. **Kou jump-diffusion** — adds asymmetric exponential jumps. Crypto exhibits this empirically: discontinuous moves, fatter downside than upside.
3. **Historical bootstrap** — resamples actual empirical returns. Preserves whatever distributional shape the real data has.
4. **Common-factor** — multi-asset paths driven by a shared (jumpy) factor. *This is the workhorse* for Phase 14.6 stress testing: the insurance fund only fails when *many* LP positions hit MaxIL together, which requires a common-factor crash.

Implementations live in `inflexion_quant/prices.py`; this notebook drives them and inspects the output.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from inflexion_quant.prices import (
    gbm_paths, kou_jump_paths, KouParams,
    bootstrap_paths,
    common_factor_paths, CommonFactor,
)
from inflexion_quant.data import synthetic_returns

rng = np.random.default_rng(seed=20260526)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.3})
print('ready')

## A. GBM baseline

`dS = μ S dt + σ S dW`. Lognormal terminal distribution; thin tails. Useful as the reference any more-elaborate model must beat.

In [ ]:
S0, mu, sigma, T = 3000.0, 0.0, 0.65, 30 / 365
n_steps, n_paths = 24 * 30, 5_000  # hourly steps, 30 days, 5k paths

g = gbm_paths(S0, mu, sigma, T, n_steps, n_paths, rng)
print(f'shape: {g.shape}  |  S0: {g[0, 0]}  |  terminal mean: {g[:, -1].mean():.1f}  |  expected: {S0 * np.exp(mu * T):.1f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
t = np.linspace(0, T * 365, n_steps + 1)
axes[0].plot(t, g[:50].T, lw=0.7, alpha=0.6)
axes[0].set(title=f'GBM — 50 sample paths (σ={sigma:.0%}, T=30d)', xlabel='days', ylabel='ETH (USD)')

r_gbm = np.log(g[:, -1] / g[:, 0])
axes[1].hist(r_gbm, bins=60, density=True, alpha=0.7, color='C0', label='simulated')
xs = np.linspace(r_gbm.min(), r_gbm.max(), 200)
axes[1].plot(xs, stats.norm.pdf(xs, loc=(mu - 0.5 * sigma**2) * T, scale=sigma * np.sqrt(T)), 'k--', lw=1.5, label='theoretical N')
axes[1].set(title='terminal log-returns', xlabel='log-return', ylabel='density')
axes[1].legend()
plt.tight_layout(); plt.show()

## B. Kou jump-diffusion

`d log S = (μ − ½σ²) dt + σ √dt Z + Σ_k Y_k`. The jump `Y` is Kou: probability `p_up` of an upward exponential jump (rate `η_up`), else a downward one (rate `η_down`). Asymmetric on purpose — set `η_down < η_up` for a fatter left tail (the crypto default).

In [ ]:
kou = KouParams(lam=80, p_up=0.4, eta_up=25, eta_down=15)  # 80 jumps/yr, fatter down tail
k = kou_jump_paths(S0, mu, sigma, T, n_steps, n_paths, kou, rng)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t, k[:50].T, lw=0.7, alpha=0.6)
axes[0].set(title=f'Kou — 50 sample paths (σ={sigma:.0%}, λ={kou.lam}/yr, T=30d)', xlabel='days', ylabel='ETH (USD)')

r_kou = np.log(k[:, -1] / k[:, 0])
bins = np.linspace(min(r_kou.min(), r_gbm.min()), max(r_kou.max(), r_gbm.max()), 80)
axes[1].hist(r_gbm, bins=bins, density=True, alpha=0.5, color='C0', label='GBM')
axes[1].hist(r_kou, bins=bins, density=True, alpha=0.5, color='C3', label='Kou')
axes[1].set(title='terminal log-returns: GBM vs Kou', xlabel='log-return', ylabel='density', yscale='log')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Excess kurtosis  GBM: {stats.kurtosis(r_gbm):+.2f}  |  Kou: {stats.kurtosis(r_kou):+.2f}  (>0 ⇒ fat tails)')
print(f'Skewness         GBM: {stats.skew(r_gbm):+.3f}  |  Kou: {stats.skew(r_kou):+.3f}  (<0 ⇒ left-skewed)')

## C. Historical bootstrap

Resamples empirical returns into forward paths. Preserves the *empirical* distribution — whatever shape the real data has (skew, kurtosis, dependence) carries through. Use this when calibrated parametric models feel restrictive.

This notebook uses **synthetic returns** from a Student-t (df=4, fat tails, calibrated to 80% annual vol) so it runs offline. To swap in real data:

```python
from inflexion_quant.data import cached_fetch, log_returns
prices = cached_fetch('ETH', days=1095)   # 3 years of daily ETH/USD from CoinGecko
empirical = log_returns(prices)
```

In [ ]:
empirical = synthetic_returns(n_days=3 * 365, rng=rng)
print(f'empirical: mean={empirical.mean():+.4f}/d, std={empirical.std():.4f}/d, '
      f'ann-vol={empirical.std() * np.sqrt(365):.0%}, '
      f'kurt={stats.kurtosis(empirical):+.2f}, skew={stats.skew(empirical):+.3f}')

n_days_fwd = 30
boot = bootstrap_paths(empirical, S0=3000.0, n_steps=n_days_fwd, n_paths=5_000, rng=rng, block_size=5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(np.arange(n_days_fwd + 1), boot[:80].T, lw=0.7, alpha=0.6)
axes[0].set(title=f'Bootstrap — 80 sample paths (block_size=5, from 3y synthetic ETH)', xlabel='days', ylabel='USD')

r_boot = np.log(boot[:, -1] / boot[:, 0])
axes[1].hist(r_boot, bins=60, density=True, alpha=0.7, color='C2')
axes[1].set(title=f'terminal log-returns ({n_days_fwd}d bootstrap)', xlabel='log-return', ylabel='density')
plt.tight_layout(); plt.show()

## D. Common-factor (the workhorse for Phase 14.6 stress)

The insurance fund only fails when *many* LP positions take losses together. That requires a **common-factor crash** — a market-wide event that moves ETH, BTC, ARB and every other LP underlying simultaneously. This model gives us controllable correlated tails.

    d log S_i  =  μ_i dt  +  β_i · dF(t)  +  σ_idio_i √dt Z_i
         dF(t) =  σ_c √dt Z_c  +  Σ_k J_k · 1{jump in dt}

For stress: ratchet `crash_lam` up and `crash_mu` deep negative.

In [ ]:
assets = ['ETH', 'BTC', 'ARB']
S0_vec = np.array([3000.0, 60_000.0, 1.20])
beta = np.array([1.10, 1.00, 1.30])              # ARB tends to amplify the market
sigma_idio = np.array([0.30, 0.25, 0.50])
cf = CommonFactor(sigma=0.55, crash_lam=4.0, crash_mu=-0.08, crash_sigma=0.03)

paths = common_factor_paths(
    S0=S0_vec, mu=np.zeros(3), sigma_idio=sigma_idio, beta=beta,
    cf=cf, T=0.5, n_steps=126, n_paths=5_000, rng=rng,
)
print(f'paths shape: {paths.shape}  (n_paths, n_assets, n_steps+1)')

r = np.log(paths[:, :, -1] / paths[:, :, 0])
corr = pd.DataFrame(np.corrcoef(r.T), index=assets, columns=assets).round(2)
print('\nTerminal log-return correlation matrix:')
print(corr)

# Pick the worst-for-ETH path: shows what a correlated drawdown looks like
worst = int(np.argmin(r[:, 0]))
ts = np.linspace(0, 365 * 0.5, paths.shape[2])
fig, ax = plt.subplots(figsize=(11, 4))
for i, name in enumerate(assets):
    ax.plot(ts, paths[worst, i, :] / paths[worst, i, 0], label=name, lw=1.4)
ax.set(title=f'Worst-for-ETH 6-month path (rescaled to 1) — correlated drawdown across all 3',
       xlabel='days', ylabel='price (rescaled)')
ax.legend(); ax.axhline(1.0, color='k', lw=0.5, ls=':')
plt.tight_layout(); plt.show()

## E. Summary — what we got and what's next

| model | use | what's still ad-hoc |
| --- | --- | --- |
| GBM | reference baseline; closed-form sanity checks | doesn't capture jumps or fat tails |
| Kou | single-asset realistic dynamics | doesn't carry cross-asset correlation |
| Bootstrap | preserves the empirical distribution exactly | only as good as the input data |
| Common-factor | multi-asset correlated stress | calibration: pick β, σ_c, crash params per market |

**Next (Task 14.3):** position-structure distribution — for each simulated price path we draw realistic LP range widths and moneyness. That's what feeds the §3.1 IL formulas in Task 14.4.

**Task 14.6 stress** then re-uses the common-factor model with extreme parameters (`σ_c +6σ`, `crash_lam` ↑, `crash_mu` deeper) and measures the insurance fund's drawdown under those scenarios. The parameters this notebook lets us tune are the same dials Phase 14.7 calibrates against ruin-probability targets.